# Generación de Queries Sintéticas para Dataset LTR
## DSRP - Curso de Ingeniería de ML

Este notebook genera queries sintéticas diversas usando:
- **Ollama (Llama 3.2:3b)** para generación creativa de queries
- **Sentence Transformers** para recuperación de candidatos basada en embeddings
- **Plantillas basadas en reglas** como queries de línea base

El objetivo es crear un dataset exploratorio que permita recomendaciones de películas diversas y de alta calidad.

In [1]:
import json
import requests
import polars as pl
import numpy as np
from sentence_transformers import SentenceTransformer
from itertools import product
from tqdm import tqdm

/Users/miguelarquezabdala/repos/dsrp-machine-learning-engineering-4/notebooks/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuración

In [2]:
# Configuración de Ollama
OLLAMA_URL = "http://localhost:11434/api/generate"
OLLAMA_MODEL = "llama3.2:3b"

# Modelo de embeddings
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

# Rutas de datos
DATA_PATH = "data/complete_imdb_database.parquet"
EMBEDDINGS_PATH = "data/movie_embs.npy"
OUTPUT_PATH = "data/ltr_synthetic_dataset.parquet"

# Parámetros LTR
TOP_K_CANDIDATES = 100  # candidatos por query
N_LABEL_BINS = 5  # etiquetas de relevancia 0-4

## 2. Cargar Datos y Modelos

In [3]:
# Cargar base de datos de películas
movies_df = pl.read_parquet(DATA_PATH)
print(f"Se cargaron {movies_df.height} películas")
print(f"Columnas: {movies_df.columns}")
movies_df.head(3)

Se cargaron 47203 películas
Columnas: ['imdb_id', 'title', 'year', 'genres', 'imdb_rating', 'imdb_votes', 'Runtime', 'Director', 'Actors', 'Plot', 'Country', 'Language']


imdb_id,title,year,genres,imdb_rating,imdb_votes,Runtime,Director,Actors,Plot,Country,Language
str,str,i32,str,f64,i64,str,str,str,str,str,str
"""tt0002423""","""Passion""",1919,"""Biography,Drama,Romance""",6.7,1105,"""113 min""","""Ernst Lubitsch""","""Pola Negri, Emil Jannings, Har…","""The story of Madame DuBarry, t…","""Germany""","""None, German"""
"""tt0004181""","""Judith of Bethulia""",1914,"""Drama""",6.2,1525,"""61 min""","""D.W. Griffith""","""Blanche Sweet, Henry B. Waltha…","""A fascinating work of high art…","""United States""","""None, English"""
"""tt0004465""","""The Perils of Pauline""",1914,"""Action,Adventure,Drama""",6.3,1116,"""199 min""","""Louis J. Gasnier, Donald MacKe…","""Pearl White, Crane Wilbur, Pau…","""Young Pauline is left a lot of…","""United States""","""None, English"""


In [4]:
# Agregar features derivadas si no están presentes
if "imdb_votes_log" not in movies_df.columns:
    movies_df = movies_df.with_columns([
        pl.col("imdb_votes").log1p().alias("imdb_votes_log"),
    ])
    
movies_df.head(3)

imdb_id,title,year,genres,imdb_rating,imdb_votes,Runtime,Director,Actors,Plot,Country,Language,imdb_votes_log
str,str,i32,str,f64,i64,str,str,str,str,str,str,f64
"""tt0002423""","""Passion""",1919,"""Biography,Drama,Romance""",6.7,1105,"""113 min""","""Ernst Lubitsch""","""Pola Negri, Emil Jannings, Har…","""The story of Madame DuBarry, t…","""Germany""","""None, German""",7.008505
"""tt0004181""","""Judith of Bethulia""",1914,"""Drama""",6.2,1525,"""61 min""","""D.W. Griffith""","""Blanche Sweet, Henry B. Waltha…","""A fascinating work of high art…","""United States""","""None, English""",7.330405
"""tt0004465""","""The Perils of Pauline""",1914,"""Action,Adventure,Drama""",6.3,1116,"""199 min""","""Louis J. Gasnier, Donald MacKe…","""Pearl White, Crane Wilbur, Pau…","""Young Pauline is left a lot of…","""United States""","""None, English""",7.018402


In [5]:
# Cargar embeddings pre-calculados de películas
movie_embs = np.load(EMBEDDINGS_PATH).astype("float32")
print(f"Dimensiones de embeddings: {movie_embs.shape}")

# Normalizar para similitud coseno
movie_norms = np.linalg.norm(movie_embs, axis=1, keepdims=True)
movie_embs_norm = movie_embs / (movie_norms + 1e-9)

Dimensiones de embeddings: (47203, 384)


In [6]:
# Cargar modelo de embeddings para codificar queries
model = SentenceTransformer(EMBEDDING_MODEL)
print(f"Modelo cargado: {EMBEDDING_MODEL}")

Modelo cargado: sentence-transformers/all-MiniLM-L6-v2


## 3. Extraer Metadatos para Generación de Queries

In [7]:
# Extraer géneros principales
genre_df = (
    movies_df
    .select(pl.col("genres").str.split(",").alias("genres_list"))
    .explode("genres_list")
    .with_columns(
        pl.col("genres_list").str.strip_chars().alias("genre")
    )
    .filter(pl.col("genre").is_not_null() & (pl.col("genre") != ""))
)

top_genres = (
    genre_df
    .group_by("genre")
    .len()
    .sort("len", descending=True)
    .head(15)
    ["genre"]
    .to_list()
)

print(f"Géneros principales: {top_genres}")

Géneros principales: ['Drama', 'Comedy', 'Action', 'Romance', 'Crime', 'Thriller', 'Horror', 'Adventure', 'Mystery', 'Fantasy', 'Biography', 'Documentary', 'Sci-Fi', 'Family', 'History']


In [8]:
# Extraer décadas
years = movies_df["year"].drop_nulls()
decades = sorted({int(y) // 10 * 10 for y in years})
print(f"Décadas: {decades}")

Décadas: [1900, 1910, 1920, 1930, 1940, 1950, 1960, 1970, 1980, 1990, 2000, 2010, 2020]


In [10]:
# Seleccionar películas populares para contexto del LLM
sample_movies = (
    movies_df
    .filter(pl.col("imdb_votes") > 50000)
    .sort("imdb_rating", descending=True)
    .head(50)
    .select(["title", "genres", "year"])
)

sample_titles = sample_movies["title"].to_list()[:20]
print(f"Películas populares de muestra: {sample_titles[:5]}")

Películas populares de muestra: ['The Shawshank Redemption', 'The Godfather', 'The Dark Knight', '12 Angry Men', 'The Godfather Part II']


## 4. Generación de Queries con Ollama

Usar Llama 3.2 para generar queries de búsqueda de películas diversas y en lenguaje natural.

In [11]:
def query_ollama(prompt: str, model: str = OLLAMA_MODEL) -> str:
    """Consultar API de Ollama y retornar texto generado."""
    payload = {
        "model": model,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": 0.8,
            "top_p": 0.9,
            "num_predict": 200,
        }
    }
    
    try:
        response = requests.post(OLLAMA_URL, json=payload, timeout=30)
        response.raise_for_status()
        return response.json().get("response", "")
    except requests.exceptions.RequestException as e:
        print(f"Error de Ollama: {e}")
        return ""


def parse_queries_from_response(response: str) -> list[str]:
    """Parsear lista numerada de queries de la respuesta del LLM."""
    queries = []
    for line in response.strip().split("\n"):
        line = line.strip()
        # Remover numeración como "1.", "1)", "- "
        if line and line[0].isdigit():
            # Remover prefijo "1. " o "1) "
            parts = line.split(".", 1) if "." in line[:3] else line.split(")", 1)
            if len(parts) > 1:
                line = parts[1].strip()
        elif line.startswith("- "):
            line = line[2:].strip()
        
        # Limpiar y validar
        line = line.strip('"\'')
        if line and len(line) > 5 and len(line) < 100:
            queries.append(line.lower())
    
    return queries

In [12]:
# Probar conexión con Ollama
test_response = query_ollama("Say 'Ollama is working' in one line.")
print(f"Prueba Ollama: {test_response if test_response else 'FALLÓ - Verifica si Ollama está corriendo'}")

Prueba Ollama: Ollama is working.


In [13]:
def generate_llm_queries(category: str, context: str, num_queries: int = 10) -> list[dict]:
    """Generar queries diversas usando LLM para una categoría específica."""
    
    prompt = f"""You are helping create a movie search dataset. Generate {num_queries} diverse, natural movie search queries for: {category}

Context: {context}

Requirements:
- Each query should be how a real user would search for movies
- Vary the phrasing (some formal, some casual)
- Include different intents: browsing, specific mood, recommendations similar to X
- Keep queries between 3-12 words
- Output ONLY the queries as a numbered list, nothing else

Examples of good queries:
- movies like inception but more emotional
- underrated sci-fi films from the 90s
- feel-good comedies for a rainy day
- intense psychological thrillers

Generate {num_queries} queries:"""

    response = query_ollama(prompt)
    queries = parse_queries_from_response(response)
    
    return [
        {
            "query_text": q,
            "intent_type": "llm_generated",
            "category": category,
            "emphasis": "neutral",
        }
        for q in queries[:num_queries]
    ]

In [14]:
# Generar queries basadas en género usando LLM
llm_queries = []

print("Generando queries basadas en género...")
for genre in tqdm(top_genres[:8]):  # Top 8 géneros
    queries = generate_llm_queries(
        category=f"{genre} movies",
        context=f"Genre: {genre}. Popular examples of this genre exist in our database.",
        num_queries=5
    )
    for q in queries:
        q["genre"] = genre
        q["decade"] = None
    llm_queries.extend(queries)

print(f"Se generaron {len(llm_queries)} queries de género")

Generando queries basadas en género...


100%|██████████| 8/8 [00:19<00:00,  2.42s/it]

Se generaron 40 queries de género


In [15]:
# Queries basadas en estado de ánimo/ocasión
print("Generando queries basadas en estado de ánimo...")
moods = [
    ("relaxing weekend", "Movies for a relaxing weekend at home"),
    ("date night", "Romantic or engaging movies for couples"),
    ("family movie night", "Family-friendly movies everyone can enjoy"),
    ("mind-bending", "Complex, thought-provoking films"),
    ("adrenaline rush", "Action-packed, exciting movies"),
    ("emotional journey", "Deep, emotionally moving films"),
    ("hidden gems", "Underrated or lesser-known quality films"),
    ("classic cinema", "Timeless classic films"),
]

for mood, context in tqdm(moods):
    queries = generate_llm_queries(
        category=mood,
        context=context,
        num_queries=5
    )
    for q in queries:
        q["genre"] = None
        q["decade"] = None
    llm_queries.extend(queries)

print(f"Total queries LLM: {len(llm_queries)}")

Generando queries basadas en estado de ánimo...


100%|██████████| 8/8 [00:19<00:00,  2.50s/it]

Total queries LLM: 80


In [16]:
# Queries tipo "Movies like X" usando películas populares de muestra
print("Generando queries de similitud...")
for title in tqdm(sample_titles[:10]):
    queries = generate_llm_queries(
        category=f"movies similar to {title}",
        context=f"Reference movie: {title}. Generate queries for finding similar movies.",
        num_queries=3
    )
    for q in queries:
        q["genre"] = None
        q["decade"] = None
        q["intent_type"] = "similarity_search"
    llm_queries.extend(queries)

print(f"Total queries LLM: {len(llm_queries)}")

Generando queries de similitud...


100%|██████████| 10/10 [00:18<00:00,  1.87s/it]

Total queries LLM: 110


## 5. Queries Basadas en Plantillas (Línea Base)

Generar queries estructuradas usando plantillas para cobertura sistemática.

In [17]:
def generate_template_queries(top_genres: list[str], decades: list[int]) -> list[dict]:
    """Generar queries usando plantillas para cobertura sistemática."""
    queries = []

    # Plantillas con marcadores de énfasis para scoring de relevancia
    templates_genre = [
        ("best {genre} movies", "genre_only", "rating"),
        ("top rated {genre} movies", "genre_only", "rating"),
        ("popular {genre} movies", "genre_only", "popularity"),
        ("classic {genre} films", "genre_only", "neutral"),
        ("must watch {genre} movies", "genre_only", "rating"),
        ("highly acclaimed {genre} movies", "genre_only", "rating"),
    ]

    # Queries solo por género
    for g in top_genres:
        for tpl, intent_type, emphasis in templates_genre:
            queries.append({
                "query_text": tpl.format(genre=g.lower()),
                "intent_type": intent_type,
                "genre": g,
                "decade": None,
                "emphasis": emphasis,
                "category": "template",
            })

    # Queries género + década
    templates_genre_decade = [
        ("best {genre} movies from the {decade}s", "genre_decade", "rating"),
        ("popular {genre} films of the {decade}s", "genre_decade", "popularity"),
        ("{decade}s {genre} classics", "genre_decade", "neutral"),
    ]

    for g, d in product(top_genres[:10], decades[-5:]):  # Últimas 5 décadas, top 10 géneros
        for tpl, intent_type, emphasis in templates_genre_decade:
            queries.append({
                "query_text": tpl.format(genre=g.lower(), decade=d),
                "intent_type": intent_type,
                "genre": g,
                "decade": d,
                "emphasis": emphasis,
                "category": "template",
            })

    # Plantillas basadas en estado de ánimo
    mood_templates = [
        ("feel good {genre} movies", "mood_feel_good"),
        ("dark {genre} movies", "mood_dark"),
        ("family friendly {genre} movies", "mood_family"),
        ("intense {genre} films", "mood_intense"),
    ]

    for g in top_genres[:10]:
        for tpl, mood_tag in mood_templates:
            queries.append({
                "query_text": tpl.format(genre=g.lower()),
                "intent_type": mood_tag,
                "genre": g,
                "decade": None,
                "emphasis": "neutral",
                "category": "template",
            })

    return queries

template_queries = generate_template_queries(top_genres, decades)
print(f"Se generaron {len(template_queries)} queries de plantilla")

Se generaron 280 queries de plantilla


## 6. Combinar y Deduplicar Queries

In [19]:
# Combinar todas las queries
all_queries = llm_queries + template_queries

# Deduplicar por query_text
seen = set()
unique_queries = []
for q in all_queries:
    text = q["query_text"].lower().strip()
    if text not in seen:
        seen.add(text)
        unique_queries.append(q)

# Asignar IDs de query
for i, q in enumerate(unique_queries, start=1):
    q["query_id"] = i

# Crear DataFrame con schema explícito para evitar errores de tipos mixtos
queries_df = pl.DataFrame(
    unique_queries,
    schema={
        "query_text": pl.Utf8,
        "intent_type": pl.Utf8,
        "category": pl.Utf8,
        "emphasis": pl.Utf8,
        "genre": pl.Utf8,
        "decade": pl.Int64,
        "query_id": pl.Int64,
    }
)
print(f"Total queries únicas: {queries_df.height}")
queries_df.head(10)

Total queries únicas: 390


query_text,intent_type,category,emphasis,genre,decade,query_id
str,str,str,str,str,i64,i64
"""best drama movies with complex…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,1
"""emotional films that make me c…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,2
"""similar to 12 years a slave bu…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,3
"""dramas that explore social ine…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,4
"""movies like a beautiful mind f…","""llm_generated""","""Drama movies""","""neutral""","""Drama""",null,5
"""funny movies to watch on a fri…","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,6
"""best comedy movies about relat…","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,7
"""movies similar to the hangover…","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,8
"""classic comedies from the 80s …","""llm_generated""","""Comedy movies""","""neutral""","""Comedy""",null,9


In [20]:
# Distribución de queries por tipo
print("\nQueries por tipo de intención:")
print(queries_df.group_by("intent_type").len().sort("len", descending=True))

print("\nQueries por categoría:")
print(queries_df.group_by("category").len().sort("len", descending=True))


Queries por tipo de intención:
shape: (8, 2)
┌───────────────────┬─────┐
│ intent_type       ┆ len │
│ ---               ┆ --- │
│ str               ┆ u32 │
╞═══════════════════╪═════╡
│ genre_decade      ┆ 150 │
│ genre_only        ┆ 90  │
│ llm_generated     ┆ 80  │
│ similarity_search ┆ 30  │
│ mood_dark         ┆ 10  │
│ mood_family       ┆ 10  │
│ mood_feel_good    ┆ 10  │
│ mood_intense      ┆ 10  │
└───────────────────┴─────┘

Queries por categoría:
shape: (27, 2)
┌─────────────────────────────────┬─────┐
│ category                        ┆ len │
│ ---                             ┆ --- │
│ str                             ┆ u32 │
╞═════════════════════════════════╪═════╡
│ template                        ┆ 280 │
│ family movie night              ┆ 5   │
│ classic cinema                  ┆ 5   │
│ emotional journey               ┆ 5   │
│ Thriller movies                 ┆ 5   │
│ …                               ┆ …   │
│ movies similar to The Lord of … ┆ 3   │
│ movies similar to

## 7. Recuperación de Candidatos

Para cada query, recuperar las top-K películas candidatas usando similitud de embeddings.

In [21]:
def get_candidates_for_query(
    q_row: dict,
    movies_df: pl.DataFrame,
    movie_embs_norm: np.ndarray,
    model: SentenceTransformer,
    k: int = TOP_K_CANDIDATES,
) -> pl.DataFrame:
    """Recuperar top-K películas candidatas para una query usando similitud de embeddings."""
    qid = q_row["query_id"]
    qtext = q_row["query_text"]

    # Codificar query
    q_emb = model.encode([qtext]).astype("float32")[0]
    q_emb = q_emb / (np.linalg.norm(q_emb) + 1e-9)

    # Calcular similitud coseno
    scores = movie_embs_norm @ q_emb

    # Obtener índices top-K
    k = min(k, scores.shape[0])
    idxs = np.argpartition(-scores, k)[:k]
    idxs = idxs[np.argsort(-scores[idxs])]

    # Construir DataFrame de candidatos
    cand = movies_df[idxs].with_columns(
        pl.Series("sim_embedding", scores[idxs].astype("float32")),
        pl.lit(qid).cast(pl.Int32).alias("query_id"),
        pl.lit(qtext).alias("query_text"),
    )

    # Seleccionar columnas
    cols = ["query_id", "query_text", "imdb_id", "title", "sim_embedding"]
    for extra in ["imdb_rating", "imdb_votes_log", "year", "genres"]:
        if extra in cand.columns:
            cols.append(extra)

    return cand.select(cols)

In [22]:
# Recuperar candidatos para todas las queries
print(f"Recuperando top-{TOP_K_CANDIDATES} candidatos para {queries_df.height} queries...")

all_candidates = []
for q_row in tqdm(queries_df.iter_rows(named=True), total=queries_df.height):
    cand = get_candidates_for_query(
        q_row=q_row,
        movies_df=movies_df,
        movie_embs_norm=movie_embs_norm,
        model=model,
        k=TOP_K_CANDIDATES,
    )
    all_candidates.append(cand)

candidates_df = pl.concat(all_candidates)
print(f"\nTotal candidatos: {candidates_df.height}")
candidates_df.head(5)

Recuperando top-100 candidatos para 390 queries...


100%|██████████| 390/390 [00:09<00:00, 41.78it/s] 


Total candidatos: 39000


query_id,query_text,imdb_id,title,sim_embedding,imdb_rating,imdb_votes_log,year,genres
i32,str,str,str,f32,f64,f64,i32,str
1,"""best drama movies with complex…","""tt0062981""","""The Girls""",0.517326,6.7,7.055313,1968,"""Comedy,Drama"""
1,"""best drama movies with complex…","""tt21440238""","""Ini Utharam""",0.51466,6.4,7.053586,2022,"""Thriller"""
1,"""best drama movies with complex…","""tt0460976""","""Vallavan""",0.513653,5.0,7.803843,2006,"""Romance,Thriller"""
1,"""best drama movies with complex…","""tt0339071""","""Girls Will Be Girls""",0.509468,6.9,7.45472,2003,"""Comedy,Romance"""
1,"""best drama movies with complex…","""tt14398454""","""Tara vs Bilal""",0.505075,5.9,8.000014,2022,"""Comedy,Drama,Romance"""


## 8. Scoring de Relevancia y Etiquetado

Calcular scores de relevancia basados en:
- Similitud de embeddings
- Rating de IMDB
- Popularidad (votos)

Los pesos varían según el énfasis de la query.

In [23]:
def add_rel_score_for_query(
    cand: pl.DataFrame,
    q_row: dict,
) -> pl.DataFrame:
    """Agregar score de relevancia basado en énfasis de la query."""
    emphasis = q_row.get("emphasis", "neutral")

    # Pesos base
    w_sim, w_rating, w_votes = 0.4, 0.4, 0.2

    # Ajustar según énfasis
    if emphasis == "rating":
        w_sim, w_rating, w_votes = 0.3, 0.5, 0.2
    elif emphasis == "popularity":
        w_sim, w_rating, w_votes = 0.3, 0.2, 0.5

    return cand.with_columns(
        (
            w_sim * pl.col("sim_embedding") +
            w_rating * (pl.col("imdb_rating") / 10.0) +
            w_votes * (pl.col("imdb_votes_log") / 15.0)
        ).alias("rel_score")
    )


def add_label_from_rel_score(cand: pl.DataFrame, n_bins: int = N_LABEL_BINS) -> pl.DataFrame:
    """Convertir rel_score continuo a etiquetas discretas (0 a n_bins-1)."""
    cand = cand.sort("rel_score", descending=True).with_row_index("rank")
    n = cand.height
    if n == 0:
        return cand

    bin_size = max(1, n // n_bins)

    # Asignar bucket basado en rank
    bucket_expr = pl.col("rank") // bin_size
    bucket_expr = pl.when(bucket_expr > (n_bins - 1)).then(n_bins - 1).otherwise(bucket_expr)

    cand = cand.with_columns(bucket_expr.alias("bucket"))

    # Invertir para que los mejores tengan la etiqueta más alta
    cand = cand.with_columns(
        (n_bins - 1 - pl.col("bucket")).cast(pl.Int32).alias("label")
    ).drop(["rank", "bucket"])

    return cand

In [24]:
# Calcular scores de relevancia y etiquetas
print("Calculando scores de relevancia y etiquetas...")

queries_by_id = {row["query_id"]: row for row in queries_df.iter_rows(named=True)}
ltr_chunks = []

for qid, q_row in tqdm(queries_by_id.items(), total=len(queries_by_id)):
    cand = candidates_df.filter(pl.col("query_id") == qid)
    if cand.is_empty():
        continue

    # Agregar score de relevancia
    cand = add_rel_score_for_query(cand, q_row)

    # Agregar etiquetas discretas
    cand = add_label_from_rel_score(cand, n_bins=N_LABEL_BINS)

    ltr_chunks.append(cand)

ltr_df = pl.concat(ltr_chunks)
print(f"\nTamaño del dataset LTR: {ltr_df.height}")

Calculando scores de relevancia y etiquetas...


100%|██████████| 390/390 [00:00<00:00, 557.58it/s]


Tamaño del dataset LTR: 39000


## 9. Dataset Final

In [25]:
# Mostrar muestra del dataset final
print(f"Forma del dataset LTR final: {ltr_df.shape}")
print(f"\nColumnas: {ltr_df.columns}")
ltr_df.head(30)

Forma del dataset LTR final: (39000, 11)

Columnas: ['query_id', 'query_text', 'imdb_id', 'title', 'sim_embedding', 'imdb_rating', 'imdb_votes_log', 'year', 'genres', 'rel_score', 'label']


query_id,query_text,imdb_id,title,sim_embedding,imdb_rating,imdb_votes_log,year,genres,rel_score,label
i32,str,str,str,f32,f64,f64,i32,str,f64,i32
1,"""best drama movies with complex…","""tt18163024""","""Chaaruseela""",0.466336,8.5,7.604396,2022,null,0.627926,4
1,"""best drama movies with complex…","""tt2243299""","""Final Cut: Ladies and Gentleme…",0.501051,8.0,8.004032,2012,"""Comedy,Drama,Romance""",0.627141,4
1,"""best drama movies with complex…","""tt0100998""","""Dreams""",0.444638,7.7,10.370048,1990,"""Drama,Fantasy""",0.624123,4
1,"""best drama movies with complex…","""tt1895484""","""Model Minority""",0.461922,8.3,7.777374,2012,"""Drama""",0.620467,4
1,"""best drama movies with complex…","""tt2245544""","""Carry on Jatta""",0.441478,8.3,8.258422,2012,"""Comedy""",0.618704,4
…,…,…,…,…,…,…,…,…,…,…
1,"""best drama movies with complex…","""tt1437366""","""Beyond""",0.450722,7.0,8.422663,2010,"""Drama""",0.572591,3
1,"""best drama movies with complex…","""tt0217629""","""Looking for Alibrandi""",0.451369,7.0,8.23695,2000,"""Comedy,Drama,Romance""",0.570373,3
1,"""best drama movies with complex…","""tt0062981""","""The Girls""",0.517326,6.7,7.055313,1968,"""Comedy,Drama""",0.569001,3


In [26]:
# Distribución de etiquetas
print("\nDistribución de etiquetas:")
print(ltr_df.group_by("label").len().sort("label"))


Distribución de etiquetas:
shape: (5, 2)
┌───────┬──────┐
│ label ┆ len  │
│ ---   ┆ ---  │
│ i32   ┆ u32  │
╞═══════╪══════╡
│ 0     ┆ 7800 │
│ 1     ┆ 7800 │
│ 2     ┆ 7800 │
│ 3     ┆ 7800 │
│ 4     ┆ 7800 │
└───────┴──────┘


In [27]:
# Muestra de una query para inspeccionar resultados
sample_qid = ltr_df["query_id"].unique()[0]
sample_query = ltr_df.filter(pl.col("query_id") == sample_qid)

print(f"Query de muestra: '{sample_query['query_text'][0]}'")
print(f"\nTop 10 resultados:")
sample_query.head(10)

Query de muestra: 'best drama movies with complex female characters'

Top 10 resultados:


query_id,query_text,imdb_id,title,sim_embedding,imdb_rating,imdb_votes_log,year,genres,rel_score,label
i32,str,str,str,f32,f64,f64,i32,str,f64,i32
1,"""best drama movies with complex…","""tt18163024""","""Chaaruseela""",0.466336,8.5,7.604396,2022,null,0.627926,4
1,"""best drama movies with complex…","""tt2243299""","""Final Cut: Ladies and Gentleme…",0.501051,8.0,8.004032,2012,"""Comedy,Drama,Romance""",0.627141,4
1,"""best drama movies with complex…","""tt0100998""","""Dreams""",0.444638,7.7,10.370048,1990,"""Drama,Fantasy""",0.624123,4
1,"""best drama movies with complex…","""tt1895484""","""Model Minority""",0.461922,8.3,7.777374,2012,"""Drama""",0.620467,4
1,"""best drama movies with complex…","""tt2245544""","""Carry on Jatta""",0.441478,8.3,8.258422,2012,"""Comedy""",0.618704,4
1,"""best drama movies with complex…","""tt0051093""","""Tokyo Twilight""",0.453766,8.0,8.624971,1957,"""Drama""",0.616506,4
1,"""best drama movies with complex…","""tt1650048""","""Laurence Anyways""",0.446079,7.6,10.043336,2012,"""Drama,Romance""",0.616343,4
1,"""best drama movies with complex…","""tt36594277""","""Cherasaala""",0.456527,8.5,6.977281,2025,"""Drama""",0.615641,4
1,"""best drama movies with complex…","""tt7797658""","""Awe!""",0.485805,7.6,8.763115,2018,"""Drama,Fantasy,Horror""",0.615164,4


In [28]:
# Guardar dataset final
ltr_df.write_parquet(OUTPUT_PATH)
print(f"Dataset LTR guardado en {OUTPUT_PATH}")
print(f"Total de queries: {ltr_df['query_id'].n_unique()}")
print(f"Total de pares query-documento: {ltr_df.height}")

Dataset LTR guardado en data/ltr_synthetic_dataset.parquet
Total de queries: 390
Total de pares query-documento: 39000


## 10. Estadísticas Resumen

In [29]:
# Estadísticas resumen
print("=" * 50)
print("RESUMEN DEL DATASET")
print("=" * 50)
print(f"Total de queries: {ltr_df['query_id'].n_unique()}")
print(f"Total de pares query-documento: {ltr_df.height}")
print(f"Candidatos por query: {TOP_K_CANDIDATES}")
print(f"Bins de etiquetas: {N_LABEL_BINS} (0-{N_LABEL_BINS-1})")
print(f"\nEstadísticas del score de relevancia:")
print(ltr_df.select(pl.col("rel_score").describe()))
print(f"\nEstadísticas de similitud de embeddings:")
print(ltr_df.select(pl.col("sim_embedding").describe()))

RESUMEN DEL DATASET
Total de queries: 390
Total de pares query-documento: 39000
Candidatos por query: 100
Bins de etiquetas: 5 (0-4)

Estadísticas del score de relevancia:


AttributeError: 'Expr' object has no attribute 'describe'